# HSBC Statements Processing

Extract and analyze transaction data from HSBC PDF statements.

In [3]:
import { saveJson, readJson } from "./utils.ts";
import {PDFParse, } from "npm:pdf-parse";
import { readdir, readFile } from "node:fs/promises";
import {join} from "node:path"
const statementsDir = "./data/hsbcStatements";

// Get all PDF files
const pdfFiles: string[] = [];
const dir = await readdir(statementsDir)
for await (const entry of dir) {
  if (entry.endsWith(".pdf")) {
    pdfFiles.push(entry);
  }
}
pdfFiles.sort();
console.log(`Found ${pdfFiles.length} PDF statements:`, pdfFiles);

Found 12 PDF statements: [
  "2025-01-16_Statement.pdf",
  "2025-02-16_Statement.pdf",
  "2025-03-16_Statement.pdf",
  "2025-04-16_Statement.pdf",
  "2025-05-16_Statement.pdf",
  "2025-06-16_Statement.pdf",
  "2025-07-16_Statement.pdf",
  "2025-08-16_Statement.pdf",
  "2025-09-16_Statement.pdf",
  "2025-10-16_Statement.pdf",
  "2025-11-16_Statement.pdf",
  "2025-12-16_Statement.pdf"
]


In [ ]:
// Extract text from a single PDF to understand the format
const samplePdf = pdfFiles[0];
const pdfPath = join(statementsDir, samplePdf);
const dataBuffer = await readFile(pdfPath);

const pdfData = new PDFParse({data: dataBuffer, })
const text = await pdfData.getText({lineThreshold: 20, disableNormalization:true})


console.log("Sample PDF text (first 3000 chars):");
console.log(text)


In [ ]:
// Transaction parsing - adjust regex based on actual PDF format
// HSBC statements typically have: Date, Description, Money Out, Money In, Balance

interface Transaction {
  date: string;
  description: string;
  moneyOut: number | null;
  moneyIn: number | null;
  balance: number | null;
  statementDate: string;
}

function parseTransactions(text: string, statementDate: string): Transaction[] {
  const transactions: Transaction[] = [];
  
  // Common HSBC date pattern: DD Mon (e.g., "15 Jan", "02 Feb")
  // Adjust this regex based on the actual PDF format you see above
  const datePattern = /^(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec))\s+/gm;
  
  // Split by lines and look for transaction patterns
  const lines = text.split('\n');
  
  for (let i = 0; i < lines.length; i++) {
    const line = lines[i].trim();
    const dateMatch = line.match(/^(\d{1,2}\s+(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec))/);
    
    if (dateMatch) {
      // Try to parse amounts - look for patterns like "1,234.56" or "123.45"
      const amountPattern = /([\d,]+\.\d{2})/g;
      const amounts = line.match(amountPattern) || [];
      
      // Extract description (text between date and amounts)
      const dateEnd = dateMatch[0].length;
      const firstAmountIndex = amounts.length > 0 ? line.indexOf(amounts[0]) : line.length;
      const description = line.slice(dateEnd, firstAmountIndex).trim();
      
      if (description && amounts.length > 0) {
        const parseAmount = (s: string) => parseFloat(s.replace(/,/g, ''));
        
        transactions.push({
          date: dateMatch[1],
          description,
          moneyOut: amounts.length >= 1 ? parseAmount(amounts[0]) : null,
          moneyIn: amounts.length >= 2 ? parseAmount(amounts[1]) : null,
          balance: amounts.length >= 3 ? parseAmount(amounts[2]) : null,
          statementDate,
        });
      }
    }
  }
  
  return transactions;
}

// Test on sample
const sampleTransactions = parseTransactions(pdfData.text, samplePdf.replace('_Statement.pdf', ''));
console.log(`Found ${sampleTransactions.length} transactions in sample`);
console.log("First 5 transactions:", sampleTransactions.slice(0, 5));